In [ ]:
# Cell 1: Tải và chuẩn bị dữ liệu tự động từ Google Drive
import os
import zipfile
import gdown

# 1. Cấu hình thư mục lưu trữ
DATA_DIR = os.path.join("..", "data")  # Nếu notebook nằm trong folder 'notebooks/'
ZIP_PATH = os.path.join(DATA_DIR, "freiburg_groceries.zip")
EXTRACT_DIR = os.path.join(DATA_DIR, "groceries")

# 2. FILE_ID lấy từ Google Drive link
GDRIVE_FILE_ID = "YOUR_GOOGLE_DRIVE_FILE_ID_HERE"  # <-- Thay ID của bạn vào đây
URL = f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}"

# 3. Tạo thư mục nếu chưa có
os.makedirs(DATA_DIR, exist_ok=True)

# 4. Kiểm tra và tải dữ liệu nếu chưa tồn tại
if not os.path.exists(EXTRACT_DIR):
    print("Dataset chưa tồn tại, bắt đầu tải từ Google Drive...")
    
    # Tải file nén nếu chưa có
    if not os.path.exists(ZIP_PATH):
        gdown.download(URL, ZIP_PATH, quiet=False)
        
    print("Đang giải nén dữ liệu...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
        
    # Xóa file zip để tiết kiệm ổ cứng (tùy chọn)
    if os.path.exists(ZIP_PATH):
        os.remove(ZIP_PATH)
        
    print(f"Hoàn thành! Dữ liệu đã sẵn sàng tại: {EXTRACT_DIR}")
else:
    print(f"Dataset đã sẵn sàng tại: {EXTRACT_DIR}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# Tự động chọn MPS (Mac Apple Silicon), CUDA (Nvidia GPU) hoặc CPU
device = torch.device(
    "mps" if torch.backends.mps.is_available() 
    else ("cuda" if torch.cuda.is_available() else "cpu")
)
print(f"Thiết bị đang sử dụng: {device}")

In [ ]:
# Image size chuẩn
IMG_SIZE = 128  # Với Shallow NN, có thể để 128x128 hoặc 224x224 (128x128 giúp train MLP nhanh hơn)

transform_pipeline = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset từ thư mục đã giải nén
dataset = datasets.ImageFolder(root=EXTRACT_DIR, transform=transform_pipeline)
class_names = dataset.classes
print(f"Tổng số ảnh: {len(dataset)} | Số lượng lớp: {len(class_names)}")

# Chia tập 80% Train - 20% Validation
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = random_split(
    dataset, [train_size, val_size], 
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False)

In [ ]:
class ShallowNN(nn.Module):
    def __init__(self, in_features, num_classes):
        super(ShallowNN, self).__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        return self.network(x)

in_features = 3 * IMG_SIZE * IMG_SIZE
model1 = ShallowNN(in_features=in_features, num_classes=len(class_names)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model1.parameters(), lr=1e-4)

print(model1)

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss, train_correct, total_train = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            train_correct += torch.sum(preds == labels)
            total_train += labels.size(0)
            
        epoch_train_loss = train_loss / total_train
        epoch_train_acc = (train_correct.double() / total_train).item()
        
        # Validation
        model.eval()
        val_loss, val_correct, total_val = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += torch.sum(preds == labels)
                total_val += labels.size(0)
                
        epoch_val_loss = val_loss / total_val
        epoch_val_acc = (val_correct.double() / total_val).item()
        
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)
        
        print(f"Epoch [{epoch+1:02d}/{epochs:02d}] "
              f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}% | "
              f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}%")
              
    return history

history = train_model(model1, train_loader, val_loader, criterion, optimizer, epochs=10)

In [ ]:
from PIL import Image

def predict_single_image(img_path, model, class_names, transform):
    model.eval()
    img = Image.open(img_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        out = model(tensor)
        probs = torch.softmax(out, dim=1)[0]
        
    top_p, top_c = torch.topk(probs, 3)
    print(f"--- Dự đoán cho ảnh: {img_path} ---")
    for i in range(3):
        print(f"Top {i+1}: {class_names[top_c[i]]} ({top_p[i]*100:.2f}%)")

# Ví dụ chạy thử với 1 ảnh trong tập validation:
# sample_img_path = val_set.dataset.samples[val_set.indices[0]][0]
# predict_single_image(sample_img_path, model1, class_names, transform_pipeline)